# 04b - SST Data Preparation (MODIS AQUA)

Processes daily MODIS AQUA sea surface temperature (SST) files into per-year netCDF files, computes a day-of-year climatology, and derives daily anomalies.

**Steps:**
1. Concatenate daily SST files by year -> `sst_daily_{year}.nc`
2. Compute day-of-year climatology across all years -> `sst_climatology_doy.nc`
3. Compute daily anomalies (SST - climatology) per year -> `sst_anomaly_daily_{year}.nc`

In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")

src_path = Path("/home/jupyter-daniela/suyana/sources/sst_peru")
out_path = Path("/home/jupyter-daniela/suyana/peru_production/features")
out_path.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2026))

/home/jupyter-daniela/.conda/envs/peru_environment/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## Step 1 - Concatenate daily files by year

In [2]:
all_files = sorted(src_path.glob("AQUA_MODIS.*.nc"))
print(f"Total files: {len(all_files)}")

files_by_year = {}
for f in all_files:
    year = int(f.name.split('.')[1][:4])
    files_by_year.setdefault(year, []).append(f)

print(f"Years found: {sorted(files_by_year.keys())}")

Total files: 8456
Years found: [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [3]:
for year in tqdm(YEARS, desc="Years"):
    out_file = out_path / f"sst_daily_{year}.nc"
    if out_file.exists():
        print(f"{year}: already exists, skipping")
        continue

    files = files_by_year.get(year, [])
    if not files:
        print(f"{year}: no files found")
        continue

    daily_arrays = []
    for f in sorted(files):
        date_str = f.name.split('.')[1]
        date = pd.Timestamp(date_str)
        try:
            ds = xr.open_dataset(f)
            sst = ds['sst'].expand_dims(time=[date])
            daily_arrays.append(sst)
            ds.close()
        except Exception as e:
            print(f"  Skipping {f.name}: {e}")

    if not daily_arrays:
        print(f"{year}: no valid files")
        continue

    ds_year = xr.concat(daily_arrays, dim='time').to_dataset(name='sst')
    ds_year = ds_year.sortby('time')
    ds_year.to_netcdf(out_file)
    print(f"{year}: {ds_year.sizes['time']} days → {out_file.name}")

Years:   0%|          | 0/11 [00:00<?, ?it/s]

Years: 100%|██████████| 11/11 [00:00<00:00, 19272.07it/s]

2015: already exists, skipping
2016: already exists, skipping
2017: already exists, skipping
2018: already exists, skipping
2019: already exists, skipping
2020: already exists, skipping
2021: already exists, skipping
2022: already exists, skipping
2023: already exists, skipping
2024: already exists, skipping
2025: already exists, skipping


## Step 2 - Compute day-of-year climatology (across all years)

In [4]:
clim_file = out_path / "sst_climatology_doy.nc"

yearly_datasets = []
for year in YEARS:
    f = out_path / f"sst_daily_{year}.nc"
    if f.exists():
        yearly_datasets.append(xr.open_dataset(f))

ds_all = xr.concat(yearly_datasets, dim='time')
ds_all = ds_all.sortby('time')

ds_clim = ds_all.groupby('time.dayofyear').mean(dim='time')
ds_clim = ds_clim.rename({'sst': 'sst_clim'})
ds_clim.to_netcdf(clim_file)

print(f"Climatology saved → {clim_file.name}")
print(ds_clim)

sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found


sh: 1: getfattr: not found
sh: 1: getfattr: not found
sh: 1: getfattr: not found


sh: 1: getfattr: not found


Climatology saved → sst_climatology_doy.nc
<xarray.Dataset> Size: 253MB
Dimensions:    (dayofyear: 366, lat: 480, lon: 360)
Coordinates:
  * lat        (lat) float32 2kB -0.02083 -0.0625 -0.1042 ... -19.94 -19.98
  * lon        (lon) float32 1kB -84.98 -84.94 -84.9 ... -70.1 -70.06 -70.02
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
Data variables:
    sst_clim   (dayofyear, lat, lon) float32 253MB 25.72 25.76 25.79 ... nan nan


## Step 3 - Compute anomalies per year (SST − climatology)

In [5]:
ds_clim = xr.open_dataset(out_path / "sst_climatology_doy.nc")

for year in tqdm(YEARS, desc="Anomalies"):
    anom_file = out_path / f"sst_anomaly_daily_{year}.nc"
    if anom_file.exists():
        print(f"{year}: already exists, skipping")
        continue

    daily_file = out_path / f"sst_daily_{year}.nc"
    if not daily_file.exists():
        print(f"{year}: daily file missing, skipping")
        continue

    ds_year = xr.open_dataset(daily_file)

    doy = ds_year['time'].dt.dayofyear
    clim_aligned = ds_clim['sst_clim'].sel(dayofyear=doy).drop_vars('dayofyear')

    anom = (ds_year['sst'] - clim_aligned).to_dataset(name='sst_anomaly')
    anom.to_netcdf(anom_file)
    ds_year.close()

    print(f"{year}: anomaly saved → {anom_file.name}")

ds_clim.close()

sh: 1: getfattr: not found


Anomalies:   0%|          | 0/11 [00:00<?, ?it/s]

2015: already exists, skipping
2016: already exists, skipping
2017: already exists, skipping
2018: already exists, skipping
2019: already exists, skipping
2020: already exists, skipping
2021: already exists, skipping
2022: already exists, skipping
2023: already exists, skipping
2024: already exists, skipping


sh: 1: getfattr: not found


Anomalies: 100%|██████████| 11/11 [00:02<00:00,  4.88it/s]

Anomalies: 100%|██████████| 11/11 [00:02<00:00,  4.87it/s]

2025: anomaly saved → sst_anomaly_daily_2025.nc
